# FlashAttention: Exact Attention at the Speed of Memory

The full code of [FlashAttention: Exact Attention at the Speed of Memory](https://residual-stream.lvtan71.workers.dev/posts/flashattention-explained/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/flashattention-explained) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/flashattention-explained.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.
Timings and memory depend on the machine; the comparison between the two methods is what carries over.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*" "torch==2.11.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "flashattention-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `traffic.py`

Counts the HBM traffic of one attention head's forward pass, standard against FlashAttention, by sequence length and row-block size, and what each keeps for the backward pass.

In [ ]:
!{sys.executable} traffic.py

The post printed:

```text
one block of Q, K and V at 128 x 128 in 16 bits: 96 KB of the 228 KB
n =   4,096: standard       132 MiB, flash      66 MiB ( 2.0x less); the score matrix alone        32 MiB per head
n =  32,768: standard     8,224 MiB, flash   4,112 MiB ( 2.0x less); the score matrix alone     2,048 MiB per head
n = 131,072: standard   131,200 MiB, flash  65,600 MiB ( 2.0x less); the score matrix alone    32,768 MiB per head
n = 32,768: flash traffic with row blocks of 64: 8,208 MiB, 128: 4,112 MiB, 256: 2,064 MiB
n =  4,096: saved for backward per head: standard      32 MiB, flash   1.0 MiB
n = 32,768: saved for backward per head: standard   2,048 MiB, flash   8.1 MiB
```

## `online_softmax.py`

The online softmax in numpy: the post's worked example block by block, then tiled attention against standard attention on random inputs.

In [ ]:
!{sys.executable} online_softmax.py

The post printed:

```text
scores [2. 1. 4. 0.] -> softmax [0.1125 0.0414 0.831  0.0152]
after block 1: max 2, sum 1.3679, output so far 1.7358 -> 1.2689
after block 2: max 4, sum 1.2034, output so far 3.3082 -> 2.7489
standard attention gives 2.7489
n = 1024, d = 64, blocks of 128: largest difference from standard attention 7.8e-16
n = 4096, d = 128, blocks of 256: largest difference from standard attention 6.8e-16
```

## `sdpa_vs_standard.py`

Times one head of three-step attention against PyTorch's fused `scaled_dot_product_attention` on the CPU at 2,048, 8,192 and 16,384 tokens, with each run's peak memory, in a fresh process per measurement.

In [ ]:
!{sys.executable} sdpa_vs_standard.py

The post printed:

```text
n =  2,048: standard     3.9 ms, peak    228 MiB;  fused     1.1 ms, peak    197 MiB;  the n x n score matrix is    16 MiB
n =  8,192: standard    44.4 ms, peak    716 MiB;  fused    14.7 ms, peak    206 MiB;  the n x n score matrix is   256 MiB
n = 16,384: standard   175.6 ms, peak  2,263 MiB;  fused    57.7 ms, peak    215 MiB;  the n x n score matrix is 1,024 MiB
largest difference between the two at n = 4,096: 6.6e-07
```